# DTCG Datacubes

In [ ]:
import dtcg
import dtcg.integration.oggm_bindings as oggm_bindings
import oggm
from oggm.core import massbalance
import json
from pathlib import Path
from datetime import datetime, date, timezone, timedelta

import numpy as np
import xarray as xr

# Setup OGGM

In [ ]:
base_url = "https://cluster.klima.uni-bremen.de/~oggm/gdirs/oggm_v1.6/L3-L5_files/2025.6/elev_bands_w_data/W5E5/per_glacier/"
output_dir = Path("./static/data/datacube_gen/")

In [ ]:
def load_ids_from_json(path: str) -> list:
    with open(path) as file:
        rgi_ids = json.load(file)
    assert isinstance(rgi_ids, list)
    return rgi_ids


iceland_ids = load_ids_from_json(path=output_dir / "vatnajokull_rgi_ids.json")
alpine_ids = load_ids_from_json(path=output_dir / "oetztal_rgi_ids.json")
rgi_ids = set(iceland_ids)
# assert len(rgi_ids) == len(iceland_ids + alpine_ids)

In [ ]:
binder = dtcg.integration.oggm_bindings.BindingsCryotempo()

In [ ]:
from oggm import workflow
from oggm.shop import w5e5


def get_data(rgi_ids: list):
    """Get dashboard data.

    Returns
    -------
    tuple
        Glacier directory, EOLIS-enhanced gridded data, and specific mass balance.
    """
    binder.init_oggm(dirname="gen-outlines", reset=True)
    gdirs = binder.get_glacier_directories(
        rgi_ids=rgi_ids, from_prepro_level=3, prepro_border=80
    )
    print("Fetching OGGM data from shop...")

    binder.get_glacier_data(gdirs=gdirs)
    # workflow.execute_entity_task(
    #         gdirs=gdirs, task=w5e5.process_w5e5_data, daily=True
    #     )
    for gdir in gdirs:
        binder.set_flowlines(gdir)
    return gdirs

In [ ]:
def export_outlines(gdir, save_path):
    glacier_outlines_gdf = gdir.read_shapefile("outlines")
    glacier_outlines_gdf.to_feather(path=save_path / "outlines.shp")

def set_save_folder(rgi_id, folder="../ext/data/l2_precompute/"):
    save_folder = Path(folder)
    assert save_folder.is_dir()
    save_path = save_folder / f"{rgi_id}/"
    Path(save_path).mkdir(parents=True, exist_ok=True)
    assert save_path.is_dir()
    return save_path


In [ ]:
# rgi_ids = ["RGI60-06.00377"]
data = {}
gdirs = get_data(rgi_ids)

In [ ]:
for gdir in gdirs:
    data[gdir.rgi_id] = {}
    output_path = set_save_folder(rgi_id=gdir.rgi_id, folder=output_dir)
    export_outlines(gdir=gdir, save_path=output_path)

In [ ]:
import geopandas as gpd
import pandas as pd
# rgi_ids = set(list(iceland_glaciers.values()))# + list(alpine_glaciers.values()))
geometries = {}
for rgi_id in rgi_ids:
    save_path= Path(output_dir / rgi_id)
    if save_path.exists():
        geometries[f"{rgi_id}"] = gpd.read_feather(save_path/"outlines.shp").to_crs(4326)

df = gpd.GeoDataFrame(pd.concat(geometries.values(), ignore_index=True), crs=list(geometries.values())[0].crs)
df.head()

In [ ]:
df = df[["RGIId", "GLIMSId","BgnDate", "EndDate", "CenLon","CenLat", "O1Region", "O2Region", "Area", "Zmax", "Zmin", "Name", "geometry"]]
df.to_feather(output_dir/f"RGI60-{df["O1Region"][0].zfill(2)}/glacier_outlines.shp")
df

In [ ]:
outlines = gpd.read_feather(output_dir/"RGI60-06/glacier_outlines.shp")
outlines

In [ ]:
simplified_outlines = outlines.copy(deep=True)

In [ ]:
def get_vertices(df):
    vertices = []
    for index, row in df.iterrows():
        vertices.append(len(list(row['geometry'].exterior.coords)))
    df["vertices"] = vertices
    print(f"Total vertices: {sum(vertices)}")
    return sum(vertices)

In [ ]:
outlines["geometry"][outlines["geometry"].is_valid == 0]

In [ ]:
import topojson as tp
topo = tp.Topology(outlines.copy(deep=True), prequantize=False)
simplified_outlines = topo.toposimplify(0.001).to_gdf().copy(deep=True)
invalid = simplified_outlines["geometry"][simplified_outlines["geometry"].is_valid == 0]
simplified_outlines.loc[invalid.index] = outlines.loc[invalid.index]

In [ ]:
invalid_simple = topo.toposimplify(0.0004).to_gdf().copy(deep=True)
invalid_simple["geometry"][108]


In [ ]:
get_vertices(invalid_simple.loc[invalid.index])
invalid_simple.loc[invalid.index]

In [ ]:
simplified_outlines.loc[invalid.index] = invalid_simple.loc[invalid.index]

In [ ]:
start_res = 0.001
while not invalid.is_empty.all() and start_res > 0:
    start_res -= 0.00005
    print("New start res:", start_res)
    invalid_simple = topo.toposimplify(start_res).to_gdf().copy(deep=True)
    simplified_outlines.loc[invalid_simple.index] = invalid_simple.loc[invalid_simple.index]
    invalid = simplified_outlines["geometry"][simplified_outlines["geometry"].is_valid == 0]
    
    

# for glacier in invalid:
#     topo.toposimplify(0.0)
invalid

In [ ]:
simplified_outlines["geometry"][simplified_outlines["geometry"].is_valid == 0]

In [ ]:
get_vertices(simplified_outlines)
get_vertices(outlines)

In [ ]:
simplified_outlines.to_feather(output_dir/f"RGI60-{df["O1Region"][0].zfill(2)}/glacier_outlines_simplified.shp")

In [ ]:
# simplified_outlines["geometry"] = outlines.copy(deep=True).simplify(tolerance=0.001, preserve_topology=True)
# simplified_outlines["geometry"][146]

In [ ]:
import dtcg.interface.plotting as dtcg_plotting
import geoviews as gv
graph_artist = dtcg_plotting.BokehMap()
overlay = gv.Polygons(outlines).opts(
    fill_color=graph_artist.palette[1],
    line_color="black",
    line_width=0.8,
    fill_alpha=0.4,
    color_index=None,
    ) * gv.Polygons(simplified_outlines).opts(
    fill_color=graph_artist.palette[2],
    line_color="black",
    line_width=0.8,
    fill_alpha=0.4,
    color_index=None,
) * gv.tile_sources.EsriWorldTopo()
figure = overlay
figure

In [ ]:
import dtcg.interface.plotting as dtcg_plotting
import geoviews as gv
graph_artist = dtcg_plotting.BokehMap()
overlay = gv.Polygons(outlines).opts(
    fill_color=graph_artist.palette[2],
    line_color="black",
    line_width=0.8,
    fill_alpha=0.4,
    color_index=None,
) * gv.tile_sources.EsriWorldTopo()
figure = overlay
figure

In [ ]:
import sys
sys.getsizeof(outlines)

In [ ]:
rgi_ids = set(alpine_ids+iceland_ids)
gdirs = get_data(rgi_ids)


In [ ]:
def get_glacier_df(rgi_ids, output_dir):
    geometries={}
    for rgi_id in rgi_ids:
        save_path= Path(output_dir / rgi_id)
        if save_path.exists():
            geometries[f"{rgi_id}"] = gpd.read_feather(save_path/"outlines.shp").to_crs(4326)

    df = gpd.GeoDataFrame(pd.concat(geometries.values(), ignore_index=True), crs=list(geometries.values())[0].crs)
    df['Name'] = np.where(df['Name'].isna() , df['RGIId'], df['Name'])

    return df

In [ ]:
def export_region_list(data, save_path="glacier_index"):
    # save_path = set_save_folder(rgi_id=save_path)
    with open(save_path / "glacier_index.json", mode="w", encoding="utf-8") as file:
        json.dump(data, file)


In [ ]:
subset = ["RGIId", "Name", "O1Region", "O2Region"]
region_list = {}
alpine_glaciers = get_glacier_df(rgi_ids=alpine_ids, output_dir=output_dir)
iceland_glaciers = get_glacier_df(rgi_ids=iceland_ids, output_dir=output_dir)

region_list["Central Europe"] = alpine_glaciers[subset].set_index("RGIId").to_dict(orient="index")
region_list["Iceland"] = iceland_glaciers[subset].set_index("RGIId").to_dict(orient="index")
export_region_list(region_list,save_path=output_dir)

In [ ]:
Path(output_dir/"RGI60-11").is_dir()